In [0]:
from pyspark.sql import functions as F

# Load tables
events = spark.read.table("default.bronze_events")
features_df = spark.read.table("default.silver_user_features")

In [0]:
label_df = events.groupBy("user_id") \
    .agg(
        F.max(
            F.when(F.col("event_type") == "purchase", 1).otherwise(0)
        ).alias("purchased")
    )

In [0]:
training_data = features_df.join(label_df, "user_id", "left")
training_data = training_data.fillna({"purchased": 0})

In [0]:
class_counts = training_data.groupBy("purchased").count().collect()

count_dict = {row["purchased"]: row["count"] for row in class_counts}
total = sum(count_dict.values())

training_data = training_data.withColumn(
    "class_weight",
    F.when(F.col("purchased") == 1,
           total / (2 * count_dict[1]))
     .otherwise(total / (2 * count_dict[0]))
)

In [0]:
train, test = training_data.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "total_events",
    "total_spent",
    "avg_price"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

train_data = assembler.transform(train)
test_data = assembler.transform(test)

train_data = train_data.select("features","purchased","class_weight")
test_data = test_data.select("features","purchased","class_weight")

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="purchased",
    weightCol="class_weight",
    maxIter=20
)

lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="purchased",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=5
)

rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="purchased",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

lr_auc = evaluator.evaluate(lr_predictions)
rf_auc = evaluator.evaluate(rf_predictions)

print(" Logistic Regression AUC:", lr_auc)
print(" Random Forest AUC:", rf_auc)

In [0]:
import matplotlib.pyplot as plt

# Your AUC values
lr_auc = 0.8368108878020637
rf_auc = 0.8296793738442957

models = ["Logistic Regression", "Random Forest"]
auc_scores = [lr_auc, rf_auc]

plt.figure()
plt.bar(models, auc_scores)

plt.xlabel("Model")
plt.ylabel("AUC Score")
plt.title("Model Comparison (AUC)")
plt.ylim(0, 1.05)

for i, v in enumerate(auc_scores):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center')

plt.show()

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="purchased",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

numTrees_list = [20, 50]
maxDepth_list = [3, 5]

best_auc = 0
best_params = None
best_model = None

for n in numTrees_list:
    for d in maxDepth_list:
        
        rf = RandomForestClassifier(
            featuresCol="features",
            labelCol="purchased",
            weightCol="class_weight",
            numTrees=n,
            maxDepth=d
        )
        
        model = rf.fit(train_data)
        predictions = model.transform(test_data)
        auc = evaluator.evaluate(predictions)
        
        print(f"numTrees={n}, maxDepth={d}, AUC={auc}")
        
        if auc > best_auc:
            best_auc = auc
            best_params = (n, d)
            best_model = model

print("\nBest AUC:", best_auc)
print("Best Parameters -> numTrees:", best_params[0], 
      ", maxDepth:", best_params[1])